# EDA Template

A reusable exploratory data analysis notebook. It follows the first half of the
end-to-end ML checklist (big picture → get data → clean → explore) and stops at the
point where modelling would start, handing over a written **Summary of Findings**.

**How to use**
1. Edit the **0.2 Config** cell (data path, target, problem type).
2. Run all cells once. Every section auto-adapts to the columns it finds.
3. Fill in the cells marked `✏️ TODO` (problem framing, dataset-specific cleaning, findings).

The demo dataset is `data/raw/toyota_corolla.csv` (used-car prices, target = `price`).
See `docs/eda_structure.md` for what each section is for.

<h1 style="color:#e34948;"><strong>0. Setup</strong></h1>

<h2 style="color:#eda100;">0.1 Imports</h2>

In [ ]:
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

<h2 style="color:#eda100;">0.2 Config</h2>

Everything dataset-specific lives here. Column names are the **snake_case** names produced in section 3.1.

In [ ]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = ROOT / "data" / "raw" / "toyota_corolla.csv"
TARGET = "price"              # set to None for unsupervised / pure exploration
PROBLEM_TYPE = "regression"   # "regression" or "classification"
ID_COLS = ["id"]              # identifiers: excluded from analysis
CAT_COLS = None               # force these to categorical; None = auto-detect
CAT_MAX_UNIQUE = 15           # numeric columns with <= this many values are treated as discrete
HIGH_CORR = 0.8               # |r| above this is flagged as multicollinearity
TOP_N_CATEGORIES = 10         # categories shown per bar chart

SAVE_FIGS = True              # write every figure to FIG_DIR (used by docs/)
FIG_DIR = ROOT / "reports" / "figures"
RANDOM_STATE = 42

<h2 style="color:#eda100;">0.3 Plot style</h2>

One consistent look: blue for single-series charts, a fixed categorical order for
hue splits (max 3 groups per chart), and a blue ↔ gray ↔ red diverging map for correlations.

In [ ]:
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
MAIN = PALETTE[0]
DIVERGING = LinearSegmentedColormap.from_list("blue_gray_red", ["#184f95", "#86b6ef", "#f0efec", "#f19a99", "#b52f2e"])

sns.set_theme(
    style="whitegrid",
    palette=PALETTE,
    rc={
        "axes.edgecolor": "#c3c2b7",
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "axes.labelcolor": "#52514e",
        "grid.color": "#ecebe7",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "lines.linewidth": 2,
        "figure.dpi": 100,
    },
)

<h2 style="color:#eda100;">0.4 Helpers</h2>

In [ ]:
def to_snake_case(df):
    """CamelCase / spaces / dashes -> snake_case column names."""
    df = df.copy()
    df.columns = (
        df.columns.str.strip()
        .str.replace(r"(?<=[a-z0-9])(?=[A-Z])", "_", regex=True)
        .str.replace(r"[\s\-/]+", "_", regex=True)
        .str.lower()
    )
    return df


def save_fig(fig, name):
    if SAVE_FIGS and name:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIG_DIR / f"{name}.png", dpi=110, bbox_inches="tight")


def plot_grid(columns, plot_fn, ncols=4, height=3.0, title=None, name=None, shared_legend=False):
    """Draw plot_fn(col, ax) for every column in a grid of subplots.

    shared_legend=True replaces per-panel legends with one legend above the grid.
    """
    columns = list(columns)
    if not columns:
        print("Nothing to plot.")
        return
    nrows = math.ceil(len(columns) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, height * nrows), squeeze=False)
    axes = axes.ravel()
    for ax, col in zip(axes, columns):
        plot_fn(col, ax)
        ax.set_title(col)
    for ax in axes[len(columns):]:
        ax.axis("off")
    fig.tight_layout()
    legend = axes[0].get_legend()
    if shared_legend and legend:
        handles, labels = legend.legend_handles, [t.get_text() for t in legend.get_texts()]
        fig.legend(handles, labels, title=legend.get_title().get_text(), loc="lower center",
                   bbox_to_anchor=(0.5, 1.0), ncol=len(labels), frameon=False)
        for ax in axes:
            if ax.get_legend():
                ax.get_legend().remove()
    if title:
        fig.suptitle(title, fontsize=13, fontweight="bold", y=1.0 + (0.7 / fig.get_figheight() if shared_legend and legend else 0))
    save_fig(fig, name)
    plt.show()


def missing_report(df):
    """Count and % of nulls per column (only columns with nulls)."""
    report = pd.DataFrame({
        "n_missing": df.isna().sum(),
        "pct_missing": df.isna().mean() * 100,
        "dtype": df.dtypes.astype(str),
    })
    return report[report.n_missing > 0].sort_values("n_missing", ascending=False)


def outlier_report(df, columns, k=1.5):
    """IQR-based outlier counts. Detection only - nothing is removed."""
    rows = []
    for col in columns:
        s = df[col].dropna()
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - k * iqr, q3 + k * iqr
        n_out = int(((s < lo) | (s > hi)).sum())
        rows.append({"column": col, "lower": lo, "upper": hi, "min": s.min(), "max": s.max(),
                     "n_outliers": n_out, "pct_outliers": 100 * n_out / len(s)})
    return pd.DataFrame(rows).set_index("column").sort_values("n_outliers", ascending=False)


def classify_columns(df, target=None, id_cols=(), cat_cols=None, cat_max_unique=15):
    """Split columns into constant / binary / categorical / discrete / continuous."""
    groups = {"constant": [], "binary": [], "categorical": [], "discrete": [], "continuous": []}
    forced_cat = set(cat_cols or [])
    for col in df.columns:
        if col in id_cols or col == target:
            continue
        n = df[col].nunique(dropna=True)
        if n <= 1:
            groups["constant"].append(col)
        elif n == 2:
            groups["binary"].append(col)
        elif col in forced_cat or not pd.api.types.is_numeric_dtype(df[col]):
            groups["categorical"].append(col)
        elif n <= cat_max_unique:
            groups["discrete"].append(col)
        else:
            groups["continuous"].append(col)
    return groups

<h1 style="color:#e34948;"><strong>1. Look at the Big Picture</strong></h1>

<h2 style="color:#eda100;">1.1 Frame the Problem</h2>

✏️ **TODO** - answer these before touching the data.

* **What is the business objective?**
  *Demo:* Predict the resale price of used Toyota Corollas so the pricing team can list cars competitively.
* **How will the output be used?**
  *Demo:* A price suggestion shown to sales staff when a trade-in is valued.
* **What does the target look like?** (regression / classification / none)
  *Demo:* `price` in EUR → regression.
* **What would a useful result be?**
  *Demo:* Suggested prices within ~€1,000 of the final sale price.

<h2 style="color:#eda100;">1.2 Questions & Hypotheses</h2>

✏️ **TODO** - write the questions this EDA must answer. Each one should map to a section below.

| # | Question / hypothesis | Answered in |
|---|---|---|
| Q1 | Is the data complete and trustworthy (nulls, duplicates, typos)? | 3, 4 |
| Q2 | What does the target's distribution look like? Does it need a transform? | 6.1 |
| Q3 | Which features move the target most? | 6.2 |
| Q4 | Are there redundant (highly correlated) features or leakage? | 6.3 |
| Q5 | Do segments (e.g. fuel type) behave differently? | 6.3 |

<h2 style="color:#eda100;">1.3 Performance Measure (for later modelling)</h2>

Pick now so the EDA looks at the right things (e.g. skewed target → consider RMSLE / log target).

* **Regression:** RMSE, MAE, MAPE, RMSLE, adjusted R²
* **Classification:** ROC-AUC, PR-AUC, F1, precision/recall, confusion matrix (check class balance in 6.1)

<h2 style="color:#eda100;">1.4 Check the Assumptions</h2>

✏️ **TODO** - list assumptions and confirm them with the data owner.

* *Demo:* Prices are final sale prices, not asking prices.
* *Demo:* `age_08_04` is the car's age in months as of August 2004.
* *Demo:* One row = one unique car sold.

<h1 style="color:#e34948;"><strong>2. Get the Data</strong></h1>

In [ ]:
raw = pd.read_csv(DATA_PATH)   # use pd.read_excel / pd.read_parquet for other formats
print(f"{raw.shape[0]:,} rows x {raw.shape[1]} columns")

In [ ]:
# Take a look at the top 5 rows
raw.head()

In [ ]:
raw.info()

In [ ]:
raw.describe(include="all").T

<h2 style="color:#eda100;">2.1 Data Dictionary</h2>

One row per column: what it means, unit, and its type. Auto-generated skeleton below; ✏️ **TODO** add meanings.

In [ ]:
data_dictionary = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "n_unique": raw.nunique(),
    "example": raw.iloc[0],
    "meaning": "",   # fill in
})
data_dictionary

<h1 style="color:#e34948;"><strong>3. Data Cleaning (General)</strong></h1>

Cleaning that is safe to do on the **whole** dataset (no information from the target or test set is used).

<h2 style="color:#eda100;">3.1 Format Column Names</h2>

In [ ]:
df = to_snake_case(raw)
df.columns.tolist()

In [ ]:
# Manual renames, if any
# df = df.rename(columns={"unnamed:_0": "id"})

<h2 style="color:#eda100;">3.2 Format Data Body</h2>

Trim whitespace in text columns, then pull structured fields out of free-text columns.

In [ ]:
text_cols = df.select_dtypes(include=["object", "string"]).columns
df[text_cols] = df[text_cols].apply(lambda s: s.str.strip())
for col in text_cols:
    print(f"{col}: {df[col].nunique()} unique -> {df[col].unique()[:8].tolist()}")

✏️ **TODO (dataset-specific)** - *Demo:* `model` is free text such as `Corolla 2.0 D4D HATCHB TERRA 2/3-Doors`.
Extract **body type** and **trim level**. Engine size is skipped: it duplicates `cc`.

In [ ]:
if "model" in df.columns:
    model_upper = df["model"].str.upper()
    body_patterns = {
        "hatchback": r"\b(?:HATCHB|HB)\b",
        "liftback": r"\b(?:LIFTB|LB|LIFT|LIFTBACK)\b",
        "sedan": r"\b(?:SEDAN|SDN)\b",
        "wagon": r"\b(?:WAGON|STATION|COMBI|WGN)\b",
        "mpv": r"\bVERSO\b",
    }
    df["body_type"] = np.select([model_upper.str.contains(p) for p in body_patterns.values()],
                                list(body_patterns), default="unknown")
    df["trim"] = (model_upper.str.extract(r"\b(TERRA|LUNA|SOL|EXEC|SPORT|G6|GL|XLI|XL)\b")[0]
                  .fillna("other").str.lower())
    display(df[["model", "body_type", "trim"]].sample(5, random_state=RANDOM_STATE))
    display(df["body_type"].value_counts().to_frame().T, df["trim"].value_counts().to_frame().T)

<h2 style="color:#eda100;">3.3 Format Datatypes</h2>

Check every column has the right type: dates → `datetime64`, flags → `int`/`bool`, labels → `category`.

In [ ]:
df.dtypes.value_counts()

In [ ]:
# Examples:
# df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
# df["is_active"] = df["is_active"].map({"Y": 1, "N": 0})
# df["grade"] = pd.Categorical(df["grade"], categories=["low", "mid", "high"], ordered=True)

<h2 style="color:#eda100;">3.4 Duplicates</h2>

In [ ]:
analysis_cols = [c for c in df.columns if c not in ID_COLS]
n_exact = df.duplicated().sum()
n_ignoring_id = df.duplicated(subset=analysis_cols).sum()
print(f"Exact duplicate rows:            {n_exact}")
print(f"Duplicates ignoring ID columns:  {n_ignoring_id}")
df[df.duplicated(subset=analysis_cols, keep=False)]

In [ ]:
# Decide: drop or keep? Duplicates ignoring the ID usually mean the same record was loaded twice.
df = df.drop_duplicates(subset=analysis_cols).reset_index(drop=True)
print(f"{len(df):,} rows after de-duplication")

<h2 style="color:#eda100;">3.5 Domain Sanity Checks</h2>

✏️ **TODO** - values that are *possible* to store but *impossible* in reality (negative ages, typos, units).
These are not statistical outliers; they are data-entry errors.

In [ ]:
# Demo: engine capacity (cc) of a Corolla is 1300-2000; 16000 is a typo for 1600.
if "cc" in df.columns:
    display(df.loc[df["cc"] > 5000, ["model", "cc"]])
    df.loc[df["cc"] > 5000, "cc"] = df.loc[df["cc"] > 5000, "cc"] // 10

<h1 style="color:#e34948;"><strong>4. Data Quality</strong></h1>

<h2 style="color:#eda100;">4.1 Missing Values</h2>

**Types of nulls** (decides how to treat them later):
* **MCAR** - missing completely at random (lost in ETL). Drop or impute.
* **MAR** - missingness explained by another observed column. Impute using that column.
* **MNAR** - missingness depends on the missing value itself (e.g. high earners hiding income). Add a "was missing" flag; model carefully.
* **By design** - field not applicable (e.g. no cancellation date for active customers). Fill with a sentinel.

**Options:** drop rows · drop column · impute (median / mode / model-based) · add an `is_missing` indicator.
*Only decide here; fit imputers on the training set after the split.*

In [ ]:
missing = missing_report(df)
print(f"{df.isna().sum().sum()} missing values in {len(missing)} columns; "
      f"{df.isna().any(axis=1).sum()} rows ({df.isna().any(axis=1).mean():.1%}) affected")
missing

In [ ]:
if len(missing):
    fig, ax = plt.subplots(figsize=(8, 0.4 * len(missing) + 1))
    ax.barh(missing.index, missing["pct_missing"], color=MAIN, height=0.6)
    ax.invert_yaxis()
    ax.set_xlabel("% missing")
    ax.set_title("Missing values by column")
    for i, v in enumerate(missing["pct_missing"]):
        ax.text(v, i, f" {v:.1f}%", va="center", color="#52514e", fontsize=9)
    save_fig(fig, "missing_values")
    plt.show()

In [ ]:
# Do nulls co-occur? (same rows missing several fields -> one upstream cause)
if len(missing) > 1:
    display(df[missing.index].isna().corr().round(2))

In [ ]:
# Rows with a missing TARGET cannot be used for supervised training
if TARGET and df[TARGET].isna().any():
    print(f"{df[TARGET].isna().sum()} rows have no {TARGET} -> drop before modelling")

<h2 style="color:#eda100;">4.2 Outliers</h2>

**Detect, don't delete.** Flag values outside `Q1 - 1.5·IQR` / `Q3 + 1.5·IQR`, then decide per column:
real but rare (keep, maybe log-transform) vs. error (fix in 3.5).

In [ ]:
num_cols = [c for c in df.select_dtypes("number").columns
            if c not in ID_COLS and df[c].nunique() > 2]
outliers = outlier_report(df, num_cols)
outliers[outliers.n_outliers > 0]

In [ ]:
plot_grid(num_cols, lambda c, ax: sns.boxplot(x=df[c], ax=ax, color=MAIN, fliersize=3, linewidth=1),
          title="Box plots (numeric columns)", name="boxplots")

<h1 style="color:#e34948;"><strong>5. Attribute Combination</strong></h1>

Create features that may explain the target better than the raw columns (ratios, rates, flags, date parts).
They go through the same EDA as everything else below.

✏️ **TODO (dataset-specific)** - *Demo:* mileage per year normalises `km` by age.

In [ ]:
if {"km", "age_08_04"}.issubset(df.columns):
    df["km_per_year"] = df["km"] / (df["age_08_04"] / 12)
if {"hp", "weight"}.issubset(df.columns):
    df["hp_per_tonne"] = df["hp"] / (df["weight"] / 1000)

<h1 style="color:#e34948;"><strong>6. EDA</strong></h1>

<h2 style="color:#eda100;">6.0 Classify Attributes</h2>

Every plot below is driven by these groups. Override with `CAT_COLS` in the config if a group looks wrong.

In [ ]:
groups = classify_columns(df, TARGET, ID_COLS, CAT_COLS, CAT_MAX_UNIQUE)
for name, cols in groups.items():
    print(f"{name:<12} ({len(cols):>2}): {cols}")

const_attribs = groups["constant"]
bin_attribs = groups["binary"]
cat_attribs = groups["categorical"]
disc_attribs = groups["discrete"]
cont_attribs = groups["continuous"]
num_attribs = disc_attribs + cont_attribs

In [ ]:
# Constant columns carry no information
if const_attribs:
    print("Dropping constant columns:", const_attribs)
    df = df.drop(columns=const_attribs)

# High-cardinality text columns (e.g. free text, raw model names) swamp plots
high_card = [c for c in cat_attribs if df[c].nunique() > 50]
print("High-cardinality categoricals (summarised, not plotted):", high_card)
cat_plot_attribs = [c for c in cat_attribs if c not in high_card]

<h2 style="color:#eda100;">6.1 Univariate</h2>

One column at a time: shape, spread, skew, rare levels.

### 6.1.1 Target

In [ ]:
if TARGET and PROBLEM_TYPE == "regression":
    y = df[TARGET].dropna()
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    sns.histplot(y, kde=True, ax=axes[0], color=MAIN)
    axes[0].set_title(f"{TARGET} (skew = {y.skew():.2f})")
    sns.histplot(np.log1p(y), kde=True, ax=axes[1], color=MAIN)
    axes[1].set_title(f"log1p({TARGET}) (skew = {np.log1p(y).skew():.2f})")
    fig.tight_layout()
    save_fig(fig, "target_distribution")
    plt.show()
    display(y.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).to_frame())
elif TARGET:
    counts = df[TARGET].value_counts()
    display(pd.DataFrame({"count": counts, "pct": counts / counts.sum() * 100}))
    fig, ax = plt.subplots(figsize=(6, 3))
    counts.plot.bar(ax=ax, color=MAIN)
    ax.set_title(f"Class balance: {TARGET}")
    save_fig(fig, "target_distribution")
    plt.show()

### 6.1.2 Numeric

In [ ]:
skew = df[num_attribs].skew().sort_values(key=abs, ascending=False).to_frame("skew")
skew["shape"] = pd.cut(skew["skew"].abs(), [-np.inf, 0.5, 1, np.inf], labels=["symmetric", "moderate", "highly skewed"])
skew

In [ ]:
plot_grid(cont_attribs, lambda c, ax: sns.histplot(df[c], kde=True, ax=ax, color=MAIN),
          title="Continuous features", name="hist_continuous")

In [ ]:
plot_grid(disc_attribs, lambda c, ax: sns.countplot(x=df[c], ax=ax, color=MAIN),
          title="Discrete features", name="count_discrete")

### 6.1.3 Binary

In [ ]:
binary_share = (df[bin_attribs]
                .apply(lambda s: s.value_counts(normalize=True).iloc[0] * 100)
                .sort_values(ascending=False)
                .to_frame("pct_majority_value"))
binary_share["near_constant"] = binary_share["pct_majority_value"] > 95
binary_share

In [ ]:
plot_grid(bin_attribs, lambda c, ax: sns.countplot(x=df[c], ax=ax, color=MAIN),
          ncols=6, height=2.4, title="Binary features", name="count_binary")

### 6.1.4 Categorical

In [ ]:
for col in high_card:
    print(f"{col}: {df[col].nunique()} levels; top 5:")
    display(df[col].value_counts().head(5).to_frame())

In [ ]:
def plot_top_categories(col, ax):
    counts = df[col].value_counts()
    top = counts.head(TOP_N_CATEGORIES)
    sns.barplot(x=top.values, y=top.index.astype(str), ax=ax, color=MAIN, orient="h")
    ax.set_xlabel("count")
    ax.set_ylabel("")

plot_grid(cat_plot_attribs, plot_top_categories, ncols=3, height=3.5,
          title="Categorical features (top levels)", name="count_categorical")

In [ ]:
# Rare levels (< 1% of rows) - candidates for grouping into "other"
rare = {c: df[c].value_counts(normalize=True).loc[lambda s: s < 0.01].index.tolist() for c in cat_plot_attribs}
{c: v for c, v in rare.items() if v}

<h2 style="color:#eda100;">6.2 Bivariate</h2>

Each feature against the **target**. This is where the candidate drivers come from.

In [ ]:
if TARGET and PROBLEM_TYPE == "regression":
    target_corr = (df[num_attribs + bin_attribs + [TARGET]]
                   .corr(method="spearman")[TARGET]
                   .drop(TARGET)
                   .sort_values(key=abs, ascending=False))
    fig, ax = plt.subplots(figsize=(7, 0.28 * len(target_corr) + 1))
    colors = np.where(target_corr > 0, PALETTE[7], PALETTE[0])
    ax.barh(target_corr.index, target_corr.values, color=colors, height=0.7)
    ax.axvline(0, color="#52514e", linewidth=1)
    ax.invert_yaxis()
    ax.set_xlabel(f"Spearman correlation with {TARGET}")
    ax.set_title(f"Features ranked by correlation with {TARGET}")
    save_fig(fig, "target_correlation")
    plt.show()

In [ ]:
if TARGET and PROBLEM_TYPE == "regression":
    top_num = [c for c in target_corr.index if c in cont_attribs][:8]
    plot_grid(top_num,
              lambda c, ax: sns.regplot(data=df, x=c, y=TARGET, ax=ax, lowess=True,
                                        scatter_kws={"s": 8, "alpha": 0.35, "color": MAIN},
                                        line_kws={"color": PALETTE[1]}),
              title=f"Top continuous features vs {TARGET} (LOWESS trend)", name="scatter_vs_target")
elif TARGET:
    plot_grid(cont_attribs,
              lambda c, ax: sns.kdeplot(data=df, x=c, hue=TARGET, common_norm=False, fill=True,
                                        alpha=0.25, linewidth=2, ax=ax),
              title=f"Continuous features by {TARGET} class", name="dist_by_target", shared_legend=True)

In [ ]:
def plot_cat_vs_target(col, ax):
    order = df[col].value_counts().head(TOP_N_CATEGORIES).index
    if pd.api.types.is_numeric_dtype(df[col]):
        order = order.sort_values()
    if PROBLEM_TYPE == "regression":
        sns.boxplot(data=df[df[col].isin(order)], x=TARGET, y=col, order=order,
                    ax=ax, color=MAIN, fliersize=2, linewidth=1, orient="h")
    else:
        rate = pd.crosstab(df[col], df[TARGET], normalize="index").loc[order]
        rate.plot.barh(stacked=True, ax=ax, width=0.7, edgecolor="white", linewidth=1)
        ax.set_xlabel("share of rows")
    ax.set_ylabel("")

if TARGET:
    by_level = cat_plot_attribs + disc_attribs + (bin_attribs if PROBLEM_TYPE == "classification" else [])
    plot_grid(by_level, plot_cat_vs_target, ncols=3, height=3.5,
              title=f"Categorical / discrete features vs {TARGET}", name="cat_vs_target", shared_legend=True)

In [ ]:
# Binary flags: difference in median target between the two groups (a = lower value, b = higher)
if TARGET and PROBLEM_TYPE == "regression":
    rows = {}
    for col in bin_attribs:
        a, b = sorted(df[col].dropna().unique())
        y_a, y_b = df.loc[df[col] == a, TARGET].dropna(), df.loc[df[col] == b, TARGET].dropna()
        rows[col] = {"a": a, "b": b, "median_when_a": y_a.median(), "median_when_b": y_b.median(),
                     "share_of_b": (df[col] == b).mean(), "p_value": stats.mannwhitneyu(y_a, y_b).pvalue}
    lift = pd.DataFrame(rows).T
    lift["diff"] = (lift["median_when_b"] - lift["median_when_a"]).astype(float)
    display(lift.sort_values("diff", key=abs, ascending=False))

<h2 style="color:#eda100;">6.3 Multivariate</h2>

Features against **each other**: redundancy, leakage, interactions.

In [ ]:
corr_cols = num_attribs + ([TARGET] if TARGET and PROBLEM_TYPE == "regression" else [])
corr = df[corr_cols].corr(method="spearman")
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(0.6 * len(corr) + 3, 0.5 * len(corr) + 2))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap=DIVERGING, vmin=-1, vmax=1, center=0,
            linewidths=2, linecolor="white", annot_kws={"size": 8}, cbar_kws={"shrink": 0.6}, ax=ax)
ax.set_title("Spearman correlation (numeric features)")
save_fig(fig, "correlation_heatmap")
plt.show()

In [ ]:
# Highly correlated pairs -> keep one of each pair (or combine) before modelling
pairs = (corr.where(np.triu(np.ones_like(corr, dtype=bool), k=1))
         .stack()
         .rename("r")
         .reset_index()
         .rename(columns={"level_0": "feature_a", "level_1": "feature_b"}))
pairs[pairs.r.abs() >= HIGH_CORR].sort_values("r", key=abs, ascending=False)

In [ ]:
# Leakage check: ID columns or row order should NOT predict the target
if TARGET and PROBLEM_TYPE == "regression":
    for col in ID_COLS:
        if col in df.columns:
            r = df[[col, TARGET]].corr(method="spearman").iloc[0, 1]
            print(f"corr({col}, {TARGET}) = {r:.2f}" + ("  <-- rows are ordered; shuffle before splitting" if abs(r) > 0.3 else ""))

### 6.3.1 Segments

✏️ **TODO** - pick a segment column (`HUE`) that might change how features relate to the target.
Hue splits use at most 3 groups; smaller groups fold into "other".

In [ ]:
HUE = "fuel_type" if "fuel_type" in df.columns else (cat_plot_attribs[0] if cat_plot_attribs else None)

if TARGET and HUE and PROBLEM_TYPE == "regression":
    keep = df[HUE].value_counts().head(3).index
    seg = df.assign(**{HUE: df[HUE].where(df[HUE].isin(keep), "other")})
    hue_order = list(keep) + (["other"] if (seg[HUE] == "other").any() else [])
    display(seg.groupby(HUE)[TARGET].describe())

    top_num = [c for c in target_corr.index if c in cont_attribs][:4]
    plot_grid(top_num,
              lambda c, ax: sns.scatterplot(data=seg, x=c, y=TARGET, hue=HUE, hue_order=hue_order,
                                            s=12, alpha=0.5, ax=ax, edgecolor="white", linewidth=0.3),
              ncols=2, height=3.5, title=f"{TARGET} vs top features, by {HUE}", name="segments", shared_legend=True)

In [ ]:
# Pairplot of the strongest features (slow on wide data - keep it small)
if TARGET and PROBLEM_TYPE == "regression":
    pp_cols = [c for c in target_corr.index if c in cont_attribs][:4] + [TARGET]
    g = sns.pairplot(df[pp_cols].dropna(), corner=True, plot_kws={"s": 8, "alpha": 0.35, "color": MAIN},
                     diag_kws={"color": MAIN})
    g.figure.suptitle("Pairplot: strongest continuous features", y=1.02)
    save_fig(g.figure, "pairplot")
    plt.show()

<h1 style="color:#e34948;"><strong>7. Summary of Findings</strong></h1>

✏️ **TODO** - this is the deliverable. Answer every question from 1.2 in one line each, then list actions.
Copy it into `docs/` when done. *Demo findings for the Toyota dataset are in `docs/findings_toyota_corolla.md`.*

**Data quality**
* ...

**Target**
* ...

**Key drivers**
* ...

**Redundancy / leakage**
* ...

**Actions before modelling**
| Action | Column(s) | Why |
|---|---|---|
| Drop rows with null target | `price` | can't train on them |
| ... | ... | ... |

<h1 style="color:#e34948;"><strong>8. Hand-off to Modelling</strong></h1>

Next steps follow the end-to-end checklist (not part of this template):

1. **Split train / test** (stratify on target bins or class; shuffle if rows are ordered).
2. **Data cleaning for modelling** - fit imputers / outlier caps on train only.
3. **Preprocess** - encode categoricals, scale numerics, build a `ColumnTransformer` pipeline.
4. **Select & train models** - baseline first, then cross-validation.
5. **Fine-tune** - grid / randomized search, ensembles.
6. **Explain** - feature importance, residual analysis.
7. **Evaluate once on the test set.**

In [ ]:
# Optional: save the cleaned dataset for the modelling notebook
# (ROOT / "data" / "processed").mkdir(parents=True, exist_ok=True)
# df.to_csv(ROOT / "data" / "processed" / "clean.csv", index=False)